# ROS 2 Architecture and Data Paths

Robot Operating System 2 (ROS 2) is a software framework rather than a separate operating system: it runs on Linux and helps programs exchange data using shared communication rules. This standalone concept lesson explains why a robot is organized as a graph of small programs and where common Duckiedrone data enters that graph. No physical device is required.

## Why modularity?

A robot may read a camera, estimate its pose, detect objects, choose a behavior, and command motors at the same time. Putting all of that work in one program quickly makes it hard to understand, test, and replace one capability without disturbing the rest.

ROS 2 organizes the same work as a graph of small programs. Each program has one focused responsibility and communicates with the rest of the robot through well-defined interfaces. A camera driver can publish images, a perception node can subscribe to them, and a controller can subscribe to the resulting pose without any one node needing to know how the others are implemented.

That separation is practical as well as tidy: nodes can run in parallel, move to another computer, or be replaced independently. On a Duckiedrone DD24, the flight controller, sensor drivers, perception nodes, and higher-level autonomy nodes can all participate in the same ROS 2 graph.

## Follow a data path

One possible robot data path looks like this:

```text
camera driver -> image topic -> perception node -> pose topic -> controller -> command interface
```

Each arrow carries a message with a defined type. The controller needs the pose message's contract, not the perception algorithm's source code. This makes it possible to test or replace one stage while keeping the rest of the graph stable. Notebook 3 examines the nodes, topics, and message contracts that make this path work.

## DTPS and ROS 2

Duckietown also uses the [Duckietown Postal Service (DTPS)](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html), a system that lets programs send data to one another. It uses Hypertext Transfer Protocol version 2 (HTTP/2)-compatible messaging, but you do not need to configure it in this learning experience (LX). DTPS and ROS 2 are separate systems: a DTPS path is not automatically a ROS 2 topic.

[`PX4`](https://docs.px4.io/main/) is the open-source autopilot firmware on the Duckiedrone's flight controller. [MAVLink](https://mavlink.io/) (Micro Air Vehicle Link) is the message protocol that connects that controller to the Duckiedrone, and [MAVROS](https://github.com/mavlink/mavros) is the bridge that translates MAVLink data into ROS 2.

Two common data paths on a Duckiedrone make that distinction concrete:

```text
camera or time-of-flight (ToF) sensor -> driver -> DTPS -> ROS 2 bridge -> ROS 2 graph -> application node
PX4 flight controller -> MAVLink -> MAVROS -> ROS 2 graph -> application node
```

Camera and ToF drivers publish through DTPS, then `ros2-camera` and `ros2-tof-bottom` republish the data for ROS 2 nodes.

Flight-controller telemetry follows the second path: MAVROS exposes PX4 data, including data from the inertial measurement unit (IMU), directly to ROS 2. An IMU measures acceleration and rotation. This is why later sensor material finds a driver and bridge pair for camera or ToF data, but no separate IMU driver on the Duckiedrone.

This learning experience focuses on ROS 2 because the nodes written here communicate through the ROS 2 graph. Later sensor learning experiences identify the sensor-specific messages and configuration on each path. The authorized, read-only remote inspection workflow appears in Notebook 8; this notebook does not require a device or issue flight-control commands.

## Further reading

The official ROS 2 Jazzy documentation on [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html) explains the streaming interface used in these data paths.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
